## Load and Preprocess Data for Fully Connected Neural Network

First, we'll import the necessary libraries. We'll use Keras to load the MNIST dataset and for building our models, and NumPy for numerical operations.

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

TensorFlow Version: 2.20.0
Keras Version: 3.13.2


Now, let's load the MNIST dataset using `keras.datasets.mnist.load_data()`. This function returns two tuples: one for training data and one for test data. Each tuple contains images and their corresponding labels.

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(f"Shape of training images (x_train): {x_train.shape}")
print(f"Shape of training labels (y_train): {y_train.shape}")
print(f"Shape of test images (x_test): {x_test.shape}")
print(f"Shape of test labels (y_test): {y_test.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Shape of training images (x_train): (60000, 28, 28)
Shape of training labels (y_train): (60000,)
Shape of test images (x_test): (10000, 28, 28)
Shape of test labels (y_test): (10000,)


For a fully connected neural network, we need to preprocess the images. This involves:
1.  **Flattening the images**: Each 28x28 image will be reshaped into a 1D array of 784 pixels.
2.  **Normalizing pixel values**: We'll convert the pixel values from integers (0-255) to floating-point numbers (0.0-1.0) by dividing by 255.0. This helps the model train more effectively.
3.  **One-hot encoding the labels**: The labels (0-9) will be converted into a binary vector representation. For example, the digit '3' will become `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`.

In [3]:
# Flatten images: convert each 28x28 image to a 784-pixel 1D array
x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))

# Normalize pixel values: convert from int (0-255) to float (0.0-1.0)
x_train_normalized = x_train_flat.astype('float32') / 255.0
x_test_normalized = x_test_flat.astype('float32') / 255.0

# One-hot encode the labels
num_classes = 10 # MNIST digits are 0-9
y_train_one_hot = keras.utils.to_categorical(y_train, num_classes)
y_test_one_hot = keras.utils.to_categorical(y_test, num_classes)

print(f"Shape of flattened and normalized training images: {x_train_normalized.shape}")
print(f"Shape of one-hot encoded training labels: {y_train_one_hot.shape}")
print(f"First 5 one-hot encoded training labels:\n{y_train_one_hot[:5]}")

Shape of flattened and normalized training images: (60000, 784)
Shape of one-hot encoded training labels: (60000, 10)
First 5 one-hot encoded training labels:
[[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


## Build and Train a Fully Connected Neural Network

We will now construct a simple fully connected neural network using Keras's `Sequential` API. The model will consist of:
1.  An input layer that matches the flattened image size (784 features).
2.  One or more `Dense` hidden layers with a ReLU activation function.
3.  An output `Dense` layer with a softmax activation function to classify the 10 digits.

In [4]:
# Define the fully connected neural network model
model_fc = keras.Sequential([
    keras.layers.Dense(units=128, activation='relu', input_shape=(784,)), # Hidden layer with 128 neurons
    keras.layers.Dense(units=64, activation='relu'), # Another hidden layer
    keras.layers.Dense(units=num_classes, activation='softmax') # Output layer with 10 neurons for 10 classes
])

# Compile the model
# Optimizer: Adam is a good general-purpose optimizer
# Loss function: categorical_crossentropy for one-hot encoded labels
# Metrics: 'accuracy' to monitor training progress
model_fc.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

model_fc.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

Now, let's train the fully connected model using our `x_train_normalized` and `y_train_one_hot` data. We will train for a few epochs and use a batch size for efficient training. The `validation_split` argument will reserve a portion of the training data for validation during training.

In [5]:
# Train the model
history_fc = model_fc.fit(x_train_normalized,
                          y_train_one_hot,
                          epochs=10, # Number of times to iterate over the entire dataset
                          batch_size=32, # Number of samples per gradient update
                          validation_split=0.1) # Use 10% of training data for validation

print("\nTraining complete. Evaluating the model on test data...")

# Evaluate the model on the test data
loss_fc, accuracy_fc = model_fc.evaluate(x_test_normalized, y_test_one_hot)

print(f"Fully Connected Model - Test Loss: {loss_fc:.4f}")
print(f"Fully Connected Model - Test Accuracy: {accuracy_fc:.4f}")

Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9265 - loss: 0.2528 - val_accuracy: 0.9645 - val_loss: 0.1170
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9671 - loss: 0.1078 - val_accuracy: 0.9755 - val_loss: 0.0809
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9766 - loss: 0.0759 - val_accuracy: 0.9768 - val_loss: 0.0727
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9824 - loss: 0.0568 - val_accuracy: 0.9772 - val_loss: 0.0766
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9854 - loss: 0.0445 - val_accuracy: 0.9788 - val_loss: 0.0817
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9880 - loss: 0.0362 - val_accuracy: 0.9798 - val_loss: 0.0763
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9901 - loss: 0.0307 - val_accuracy: 0.9822 - val_loss: 0.0695
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9917 - loss: 0.0248 - 

## Preprocess Data for a Convolutional Neural Network (CNN)

For CNNs, the input data needs to have a channel dimension. Since our MNIST images are grayscale, they have a single channel. We need to reshape our `x_train` and `x_test` data from `(num_samples, 28, 28)` to `(num_samples, 28, 28, 1)`.

We also need to ensure the pixel values are normalized to a range between 0 and 1, which we've already done for the fully connected network, but we'll re-apply it here for clarity with the reshaped data.

In [6]:
# Reshape data to add a channel dimension for CNNs
# For grayscale images, the channel dimension is 1
x_train_cnn = x_train.reshape((x_train.shape[0], 28, 28, 1))
x_test_cnn = x_test.reshape((x_test.shape[0], 28, 28, 1))

# Normalize pixel values to 0-1
x_train_cnn = x_train_cnn.astype('float32') / 255.0
x_test_cnn = x_test_cnn.astype('float32') / 255.0

# One-hot encode the labels (already done, but re-assigning for clarity for CNN data)
y_train_one_hot_cnn = y_train_one_hot
y_test_one_hot_cnn = y_test_one_hot

print(f"Shape of CNN training images: {x_train_cnn.shape}")
print(f"Shape of CNN test images: {x_test_cnn.shape}")

Shape of CNN training images: (60000, 28, 28, 1)
Shape of CNN test images: (10000, 28, 28, 1)


## Build and Train a Convolutional Neural Network (CNN)

Now we'll build a CNN. A typical CNN architecture includes `Conv2D` layers for feature extraction, `MaxPooling2D` layers for downsampling, a `Flatten` layer to convert the 2D feature maps into a 1D vector, and then `Dense` layers for classification, similar to the fully connected network.

In [7]:
# Define the CNN model
model_cnn = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(num_classes, activation='softmax')
])

# Compile the model
model_cnn.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

model_cnn.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

Let's train the CNN model using the preprocessed CNN training data. We'll use similar training parameters (epochs, batch size, validation split) to ensure a fair comparison with the fully connected model.

In [8]:
# Train the CNN model
history_cnn = model_cnn.fit(x_train_cnn,
                            y_train_one_hot_cnn,
                            epochs=10,
                            batch_size=32,
                            validation_split=0.1)

print("\nCNN Training complete. Evaluating the model on test data...")

# Evaluate the CNN model on the test data
loss_cnn, accuracy_cnn = model_cnn.evaluate(x_test_cnn, y_test_one_hot_cnn)

print(f"Convolutional Neural Network - Test Loss: {loss_cnn:.4f}")
print(f"Convolutional Neural Network - Test Accuracy: {accuracy_cnn:.4f}")

Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9525 - loss: 0.1538 - val_accuracy: 0.9847 - val_loss: 0.0513
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9846 - loss: 0.0492 - val_accuracy: 0.9883 - val_loss: 0.0406
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 30s 18ms/step - accuracy: 0.9891 - loss: 0.0336 - val_accuracy: 0.9892 - val_loss: 0.0394
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9918 - loss: 0.0251 - val_accuracy: 0.9895 - val_loss: 0.0333
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 31s 18ms/step - accuracy: 0.9945 - loss: 0.0182 - val_accuracy: 0.9902 - val_loss: 0.0361
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9953 - loss: 0.0145 - val_accuracy: 0.9907 - val_loss: 0.0373
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9964 - loss: 0.0107 - val_accuracy: 0.9893 - val_loss: 0.0452
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 30s 18ms/step - accuracy: 0.9963 -

## Compare Model Performances

Finally, let's compare the test accuracies of both the fully connected neural network and the convolutional neural network to observe the impact of different architectures.

In [9]:
print(f"Summary of Model Performances:")
print(f"Fully Connected Model Test Accuracy: {accuracy_fc:.4f}")
print(f"Convolutional Neural Network Test Accuracy: {accuracy_cnn:.4f}")

if accuracy_cnn > accuracy_fc:
    print("The CNN model performed better than the Fully Connected model.")
elif accuracy_fc > accuracy_cnn:
    print("The Fully Connected model performed better than the CNN model (this is less common for image tasks).")
else:
    print("Both models performed similarly.")

Summary of Model Performances:
Fully Connected Model Test Accuracy: 0.9752
Convolutional Neural Network Test Accuracy: 0.9919
The CNN model performed better than the Fully Connected model.
